# Training Dataset Builder — Weather-Enriched

Extends `training.ipynb` with weather data from [Open-Meteo](https://open-meteo.com/).

**Additional features over the baseline model:**

| Group | Feature | Description |
|---|---|---|
| Vessel-position | `wave_height` | Significant wave height (m) |
| Vessel-position | `swell_wave_height` | Swell height independent of local wind waves (m) |
| Vessel-position | `ocean_current_velocity` | Ocean current speed (m/s) |
| Vessel-position | `wind_speed_10m` | 10 m wind speed (m/s) |
| Vessel-position | `headwind_component` | `wind_speed × cos(cog − wind_dir)` — positive = headwind |
| Vessel-position | `sea_state` | Beaufort sea state (0–8) derived from wave height |
| Vessel-position | `adverse_sea_state` | Binary: wave ≥ 2 m or wind ≥ 10 m/s |
| Destination-port | `port_wind_gusts` | Max gust at destination port (m/s) |
| Destination-port | `port_precipitation` | Precipitation at destination port (mm) |
| Destination-port | `port_high_wind` | Binary: port gusts ≥ 15 m/s (berthing-delay threshold) |
| Destination-port | `port_precip_flag` | Binary: precipitation event at port |

All weather data is fetched from Open-Meteo (ERA5 reanalysis) and cached to
`data/cache/weather/` so subsequent runs are instant.

**Run `training.ipynb` first** to generate `data/cache/labeled.parquet` and
`data/cache/voyages.parquet`.

## 0. Imports

In [ ]:
import sys
import time
sys.path.insert(0, '..')
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import geopandas as gpd
import requests
import matplotlib.pyplot as plt
from shapely.geometry import Point

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
import joblib

from src.methods import dms_to_dd
from src.voyage_creator import VoyageCreator

pd.set_option('display.max_columns', 30)
%matplotlib inline

## 1. Load AIS & port data

In [ ]:
folder = Path("../data/ais")

df_ais = pd.concat(
    [pq.read_table(f, use_pandas_metadata=False).to_pandas()
     for f in sorted(folder.glob("ais-2025-01-0*.parquet"))],
    ignore_index=True,
)

df_cargo = df_ais[df_ais["vessel_type"].between(70, 79)].copy()
df_cargo["base_date_time"] = pd.to_datetime(df_cargo["base_date_time"])
df_cargo = df_cargo.sort_values(["mmsi", "base_date_time"]).reset_index(drop=True)

df_ports = pd.read_csv("../data/ports/ports.csv")
df_ports["lat_dd"] = df_ports["latitude"].apply(dms_to_dd)
df_ports["lon_dd"] = df_ports["longitude"].apply(dms_to_dd)
df_ports["geometry"] = df_ports.apply(lambda r: Point(r["lon_dd"], r["lat_dd"]), axis=1)
gdf_ports = gpd.GeoDataFrame(df_ports, geometry="geometry", crs="EPSG:4326")

print(f"Cargo pings : {len(df_cargo):,}")
print(f"Ports       : {len(gdf_ports):,}")

## 2. Run the voyage pipeline

Detect port visits, label every ping, and build voyage records.
Loads from `data/cache/` if already generated by `training.ipynb`.

In [ ]:
cache_dir    = Path("../data/cache")
labeled_path = cache_dir / "labeled.parquet"
voyages_path = cache_dir / "voyages.parquet"

if labeled_path.exists() and voyages_path.exists():
    df_labeled = pq.read_table(labeled_path, use_pandas_metadata=False).to_pandas()
    df_labeled["base_date_time"] = pd.to_datetime(df_labeled["base_date_time"])

    df_voyages = pq.read_table(voyages_path, use_pandas_metadata=False).to_pandas()
    df_voyages["departure_time"] = pd.to_datetime(df_voyages["departure_time"])
    df_voyages["arrival_time"]   = pd.to_datetime(df_voyages["arrival_time"])

    print("Loaded from cache.")
else:
    cache_dir.mkdir(parents=True, exist_ok=True)
    creator = VoyageCreator(gdf_ports, radius_nm=10, max_speed_knots=1.5, gap_threshold_h=24)

    port_visits            = creator.find_port_visits(df_cargo)
    df_labeled             = creator.label_pings(df_cargo, port_visits)
    df_labeled, df_voyages = creator.build_voyages(df_labeled, port_visits)

    df_labeled.to_parquet(labeled_path, index=False)
    df_voyages.to_parquet(voyages_path, index=False)
    print("Pipeline complete and cached.")

print(f"Voyages       : {len(df_voyages):,}")
print(f"Labeled pings : {len(df_labeled):,}")

## 3. Narrow to sea pings inside a detected voyage

In [ ]:
port_loc = gdf_ports.set_index("portName")[["lat_dd", "lon_dd"]]

df_sea = df_labeled[
    df_labeled["voyage_id"].notna() & df_labeled["destination_port"].notna()
].copy()
df_sea["voyage_id"] = df_sea["voyage_id"].astype(int)

df_sea = df_sea.merge(
    df_voyages[["voyage_id", "arrival_time"]],
    on="voyage_id",
    how="left",
)

# Join destination coordinates
df_sea = df_sea.join(
    port_loc.rename(columns={"lat_dd": "dest_lat", "lon_dd": "dest_lon"}),
    on="destination_port",
)
# Join origin coordinates
df_sea = df_sea.join(
    port_loc.rename(columns={"lat_dd": "origin_lat", "lon_dd": "origin_lon"}),
    on="origin_port",
)

print(f"Sea pings with voyage context : {len(df_sea):,}")
print(f"Unique voyages                : {df_sea['voyage_id'].nunique():,}")
print(f"Unique vessels                : {df_sea['mmsi'].nunique():,}")

## 4. Weather fetching setup

AIS pings arrive every 1–6 minutes, but meteorological reanalysis data is **hourly**.
We use the same efficient grid-binning strategy as `weather_ais_exploration.ipynb`:

1. **Round** vessel positions to a 1° grid (~60 nm resolution).
2. **One API call per grid cell** returning the full hourly time series.
3. **Floor** AIS timestamps to the hour and **join** on `(lat_bin, lon_bin, hour)`.

All responses are cached to `data/cache/weather/`.

In [ ]:
WEATHER_CACHE = Path("../data/cache/weather")
WEATHER_CACHE.mkdir(parents=True, exist_ok=True)

DATE_START = df_sea["base_date_time"].min().date().isoformat()
DATE_END   = df_sea["base_date_time"].max().date().isoformat()

print(f"Date range: {DATE_START} → {DATE_END}")


def _get(url: str, params: dict, retries: int = 3) -> dict:
    """GET request with exponential back-off retry."""
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r.json()
        except requests.RequestException as exc:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)


def fetch_marine_weather(lat: float, lon: float, start: str, end: str) -> pd.DataFrame:
    """Hourly marine weather from Open-Meteo Marine API (ERA5 ocean).

    Returns NaN for inland / river locations not covered by the ocean model.
    """
    data = _get(
        "https://marine-api.open-meteo.com/v1/marine",
        params=dict(
            latitude=lat, longitude=lon,
            hourly=",".join([
                "wave_height", "wave_direction", "wave_period",
                "swell_wave_height", "ocean_current_velocity",
                "ocean_current_direction",
            ]),
            start_date=start, end_date=end,
            timezone="UTC",
        ),
    )
    h = data["hourly"]
    return pd.DataFrame({
        "time":                    pd.to_datetime(h["time"]),
        "wave_height":             h["wave_height"],
        "wave_period":             h["wave_period"],
        "swell_wave_height":       h["swell_wave_height"],
        "ocean_current_velocity":  h["ocean_current_velocity"],
    })


def fetch_surface_weather(lat: float, lon: float, start: str, end: str) -> pd.DataFrame:
    """Hourly surface weather from Open-Meteo Archive API (ERA5).

    Available for all land and sea locations including rivers and ports.
    """
    data = _get(
        "https://archive-api.open-meteo.com/v1/archive",
        params=dict(
            latitude=lat, longitude=lon,
            hourly=",".join([
                "wind_speed_10m", "wind_gusts_10m", "wind_direction_10m",
                "precipitation", "weather_code", "visibility",
            ]),
            wind_speed_unit="ms",
            start_date=start, end_date=end,
            timezone="UTC",
        ),
    )
    h = data["hourly"]
    return pd.DataFrame({
        "time":               pd.to_datetime(h["time"]),
        "wind_speed_10m":     h["wind_speed_10m"],
        "wind_gusts_10m":     h["wind_gusts_10m"],
        "wind_direction_10m": h["wind_direction_10m"],
        "precipitation":      h["precipitation"],
        "visibility":         pd.array(h.get("visibility", [None] * len(h["time"])),
                                       dtype="Float64"),
    })


# Bin vessel positions for grid-level fetching
df_sea["lat_bin"] = df_sea["latitude"].round(0)
df_sea["lon_bin"] = df_sea["longitude"].round(0)
df_sea["hour"]    = df_sea["base_date_time"].dt.floor("h")

location_bins = df_sea[["lat_bin", "lon_bin"]].drop_duplicates().reset_index(drop=True)
print(f"Unique 1° grid cells: {len(location_bins)}")

## 5. Fetch vessel-position marine weather (waves, swell, current)

In [ ]:
marine_dfs = []

for _, row in location_bins.iterrows():
    lat, lon   = row["lat_bin"], row["lon_bin"]
    cache_file = WEATHER_CACHE / f"marine_{lat:.0f}_{lon:.0f}.parquet"

    if cache_file.exists():
        df_w = pd.read_parquet(cache_file)
    else:
        try:
            df_w = fetch_marine_weather(lat, lon, DATE_START, DATE_END)
            df_w["lat_bin"] = lat
            df_w["lon_bin"] = lon
            df_w.to_parquet(cache_file, index=False)
            time.sleep(0.15)
        except Exception as exc:
            print(f"  Marine API failed for ({lat}, {lon}): {exc}")
            continue

    marine_dfs.append(df_w)

df_marine = pd.concat(marine_dfs, ignore_index=True)
df_marine["time"] = pd.to_datetime(df_marine["time"])
print(f"Marine weather records: {len(df_marine):,}")

## 6. Fetch vessel-position surface wind

In [ ]:
wind_dfs = []

for _, row in location_bins.iterrows():
    lat, lon   = row["lat_bin"], row["lon_bin"]
    cache_file = WEATHER_CACHE / f"wind_{lat:.0f}_{lon:.0f}.parquet"

    if cache_file.exists():
        df_w = pd.read_parquet(cache_file)
    else:
        try:
            df_w = fetch_surface_weather(lat, lon, DATE_START, DATE_END)
            df_w["lat_bin"] = lat
            df_w["lon_bin"] = lon
            df_w.to_parquet(cache_file, index=False)
            time.sleep(0.15)
        except Exception as exc:
            print(f"  Wind API failed for ({lat}, {lon}): {exc}")
            continue

    wind_dfs.append(df_w)

df_wind = pd.concat(wind_dfs, ignore_index=True)
df_wind["time"] = pd.to_datetime(df_wind["time"])
print(f"Wind weather records: {len(df_wind):,}")

## 7. Merge vessel-position weather onto sea pings

In [ ]:
df_sea = df_sea.merge(
    df_marine.rename(columns={"time": "hour"}),
    on=["lat_bin", "lon_bin", "hour"],
    how="left",
)

df_sea = df_sea.merge(
    df_wind.rename(columns={"time": "hour"}),
    on=["lat_bin", "lon_bin", "hour"],
    how="left",
)

marine_cov = df_sea["wave_height"].notna().mean()
wind_cov   = df_sea["wind_speed_10m"].notna().mean()
print(f"Marine coverage : {marine_cov:.1%}")
print(f"Wind coverage   : {wind_cov:.1%}")

## 8. Fetch destination-port weather

Port conditions (wind, gusts, precipitation) affect berthing and pilotage.
One API call per unique destination port.

In [ ]:
dest_ports = (
    df_sea[["destination_port", "dest_lat", "dest_lon"]]
    .drop_duplicates("destination_port")
    .dropna(subset=["dest_lat", "dest_lon"])
    .reset_index(drop=True)
)
print(f"Unique destination ports: {len(dest_ports)}")

port_weather_dfs = []

for _, row in dest_ports.iterrows():
    port_name  = row["destination_port"]
    lat, lon   = row["dest_lat"], row["dest_lon"]
    safe_name  = port_name.replace("/", "_").replace(" ", "_")
    cache_file = WEATHER_CACHE / f"port_{safe_name}.parquet"

    if cache_file.exists():
        df_pw = pd.read_parquet(cache_file)
    else:
        try:
            df_pw = fetch_surface_weather(lat, lon, DATE_START, DATE_END)
            df_pw["destination_port"] = port_name
            df_pw.to_parquet(cache_file, index=False)
            time.sleep(0.15)
        except Exception as exc:
            print(f"  Port weather failed for {port_name}: {exc}")
            continue

    port_weather_dfs.append(df_pw)

df_port_weather = pd.concat(port_weather_dfs, ignore_index=True)
df_port_weather["time"] = pd.to_datetime(df_port_weather["time"])

rename_map = {
    "wind_speed_10m":     "port_wind_speed",
    "wind_gusts_10m":     "port_wind_gusts",
    "wind_direction_10m": "port_wind_direction",
    "precipitation":      "port_precipitation",
    "weather_code":       "port_weather_code",
    "visibility":         "port_visibility",
}
df_port_weather = df_port_weather.rename(columns=rename_map)

df_sea = df_sea.merge(
    df_port_weather.rename(columns={"time": "hour"}),
    on=["destination_port", "hour"],
    how="left",
)

port_cov = df_sea["port_wind_gusts"].notna().mean()
print(f"Port weather coverage: {port_cov:.1%}")

## 9. Derived weather features

| Feature | Formula | Interpretation |
|---|---|---|
| `headwind_component` | `wind_speed × cos(cog − wind_dir)` | >0 = headwind slowing vessel; <0 = tailwind |
| `sea_state` | Beaufort scale from wave height | Ordinal severity 0–8 |
| `adverse_sea_state` | `wave_height ≥ 2 m OR wind ≥ 10 m/s` | Binary rough-conditions flag |
| `port_high_wind` | `port_wind_gusts ≥ 15 m/s` | Binary berthing-delay threshold |
| `port_precip_flag` | `port_precipitation > 0` | Binary precipitation event |

In [ ]:
def wave_height_to_beaufort(h: pd.Series) -> pd.Series:
    """Map significant wave height (m) to approximate Beaufort sea state (0–8)."""
    h_float = pd.to_numeric(h, errors="coerce")  # coerce None/NaN to float NaN
    bins    = [-np.inf, 0.1, 0.5, 1.25, 2.5, 4.0, 6.0, 9.0, 14.0, np.inf]
    labels  = [0, 1, 2, 3, 4, 5, 6, 7, 8]
    return pd.cut(h_float, bins=bins, labels=labels, right=True).astype("Int64")


# Headwind component (positive = vessel heading into the wind)
angle_diff = (df_sea["cog"] - df_sea["wind_direction_10m"]) % 360
df_sea["headwind_component"] = (
    np.cos(np.radians(angle_diff.astype(float))) * df_sea["wind_speed_10m"].astype(float)
)

# Beaufort sea state from wave height
df_sea["sea_state"] = wave_height_to_beaufort(df_sea["wave_height"])

# Binary adverse conditions
wave_ok  = pd.to_numeric(df_sea["wave_height"], errors="coerce")
wind_ok  = pd.to_numeric(df_sea["wind_speed_10m"], errors="coerce")
gusts_ok = pd.to_numeric(df_sea["port_wind_gusts"], errors="coerce")
precip_ok = pd.to_numeric(df_sea["port_precipitation"], errors="coerce")

df_sea["adverse_sea_state"] = ((wave_ok >= 2.0) | (wind_ok >= 10.0)).astype("Int8")
df_sea["port_high_wind"]    = (gusts_ok >= 15.0).astype("Int8")
df_sea["port_precip_flag"]  = (precip_ok > 0).astype("Int8")

derived_cols = [
    "headwind_component", "sea_state", "adverse_sea_state",
    "port_high_wind", "port_precip_flag",
]
print("Derived weather feature summary:")
display(df_sea[derived_cols].describe().round(3))

## 10. Remaining distance & fraction of voyage completed

In [ ]:
def haversine_nm(lat1, lon1, lat2, lon2):
    """Vectorised great-circle distance in nautical miles."""
    radius = 3440.065
    phi1, phi2   = np.radians(lat1), np.radians(lat2)
    delta_phi    = np.radians(lat2 - lat1)
    delta_lam    = np.radians(lon2 - lon1)
    a = (np.sin(delta_phi / 2) ** 2
         + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lam / 2) ** 2)
    return 2 * radius * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


df_sea["remaining_dist_nm"] = haversine_nm(
    df_sea["latitude"],  df_sea["longitude"],
    df_sea["dest_lat"],  df_sea["dest_lon"],
)

df_sea["total_dist_nm"] = haversine_nm(
    df_sea["origin_lat"], df_sea["origin_lon"],
    df_sea["dest_lat"],   df_sea["dest_lon"],
)

df_sea["fraction_completed"] = (
    1 - df_sea["remaining_dist_nm"] / df_sea["total_dist_nm"].replace(0, np.nan)
).clip(0, 1)

## 11. Rolling average SOG (6 h and 24 h)

In [ ]:
df_sea = df_sea.sort_values(["mmsi", "base_date_time"]).set_index("base_date_time")

sog_grp = df_sea.groupby("mmsi")["sog"]
df_sea["sog_roll_6h"]  = sog_grp.rolling("6h").mean().reset_index(level=0, drop=True)
df_sea["sog_roll_24h"] = sog_grp.rolling("24h").mean().reset_index(level=0, drop=True)

df_sea = df_sea.reset_index()

## 12. Speed deviation

In [ ]:
LANE_COLS = ["origin_port", "destination_port"]

df_sea["vessel_lane_avg_sog"] = df_sea.groupby(["mmsi"] + LANE_COLS)["sog"].transform("mean")
df_sea["lane_avg_sog"]        = df_sea.groupby(LANE_COLS)["sog"].transform("mean")

df_sea["hist_avg_sog"]  = df_sea["vessel_lane_avg_sog"].fillna(df_sea["lane_avg_sog"])
df_sea["sog_deviation"] = df_sea["sog"] - df_sea["hist_avg_sog"]
df_sea = df_sea.drop(columns=["vessel_lane_avg_sog", "lane_avg_sog"])

## 13. Label: remaining travel time

In [ ]:
df_sea["remaining_travel_hours"] = (
    (df_sea["arrival_time"] - df_sea["base_date_time"]).dt.total_seconds() / 3600
)

n_neg = (df_sea["remaining_travel_hours"] < 0).sum()
if n_neg:
    print(f"Dropping {n_neg:,} pings with negative remaining travel time")
    df_sea = df_sea[df_sea["remaining_travel_hours"] >= 0]

print(f"Final sea-ping count: {len(df_sea):,}")
print()
print("Label statistics (remaining_travel_hours):")
display(df_sea["remaining_travel_hours"].describe().round(2))

## 14. Feature set summary

In [ ]:
# Baseline kinematic features (same as training.ipynb)
BASELINE_NUM = [
    "sog", "sog_roll_6h", "sog_roll_24h", "sog_deviation",
    "remaining_dist_nm", "total_dist_nm", "fraction_completed",
]
BASELINE_CAT = ["origin_port", "destination_port"]

# Weather features identified in weather_ais_exploration.ipynb
WEATHER_NUM = [
    "wave_height", "swell_wave_height", "ocean_current_velocity",
    "wind_speed_10m", "headwind_component",
    "port_wind_gusts", "port_precipitation",
]
WEATHER_BIN = [
    "adverse_sea_state", "port_high_wind", "port_precip_flag",
]
WEATHER_ORD = ["sea_state"]  # ordinal 0–8; treated as numeric

ALL_NUM = BASELINE_NUM + WEATHER_NUM + WEATHER_BIN + WEATHER_ORD
ALL_CAT = BASELINE_CAT
TARGET  = "remaining_travel_hours"

df_model = df_sea[ALL_NUM + ALL_CAT + ["voyage_id", TARGET]].dropna(subset=[TARGET])

print(f"Rows : {len(df_model):,}")
print(f"Null rates for weather features:")
display(
    df_model[WEATHER_NUM + WEATHER_BIN + WEATHER_ORD]
    .isna().mean().rename("null_rate")
    .to_frame()
    .style.format("{:.1%}")
)

## 15. Train / test split by voyage

In [ ]:
voyage_ids = df_model["voyage_id"].unique()
train_ids, test_ids = train_test_split(voyage_ids, test_size=0.2, random_state=42)

train_df = df_model[df_model["voyage_id"].isin(train_ids)]
test_df  = df_model[df_model["voyage_id"].isin(test_ids)]

print(f"Train : {len(train_df):,} pings  ({len(train_ids)} voyages)")
print(f"Test  : {len(test_df):,} pings  ({len(test_ids)} voyages)")

## 16. Baseline model (kinematic features only)

Reproduces `training.ipynb` to establish a performance reference.

In [ ]:
cat_indices_base = list(range(len(BASELINE_NUM), len(BASELINE_NUM) + len(BASELINE_CAT)))

preprocessor_base = ColumnTransformer([
    ("num", "passthrough", BASELINE_NUM),
    ("cat", OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=np.nan,
    ), BASELINE_CAT),
])

pipeline_base = Pipeline([
    ("preprocessor", preprocessor_base),
    ("regressor", HistGradientBoostingRegressor(
        max_iter=200, max_depth=5, learning_rate=0.05,
        categorical_features=cat_indices_base,
        random_state=42,
    )),
])

model_base = TransformedTargetRegressor(
    regressor=pipeline_base,
    func=np.log1p,
    inverse_func=np.expm1,
)

X_train_base = train_df[BASELINE_NUM + BASELINE_CAT]
y_train      = train_df[TARGET]
X_test_base  = test_df[BASELINE_NUM + BASELINE_CAT]
y_test       = test_df[TARGET]

model_base.fit(X_train_base, y_train)

y_pred_base = model_base.predict(X_test_base)
mae_base    = mean_absolute_error(y_test, y_pred_base)
rmse_base   = np.sqrt(mean_squared_error(y_test, y_pred_base))
r2_base     = r2_score(y_test, y_pred_base)

print(f"Baseline  →  MAE: {mae_base:.2f} h   RMSE: {rmse_base:.2f} h   R²: {r2_base:.3f}")

## 17. Weather-enriched model

`HistGradientBoostingRegressor` handles `NaN` values natively, so inland waterway pings
without wave data simply do not split on those features.

In [ ]:
cat_indices_wx = list(range(len(ALL_NUM), len(ALL_NUM) + len(ALL_CAT)))

preprocessor_wx = ColumnTransformer([
    ("num", "passthrough", ALL_NUM),
    ("cat", OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=np.nan,
    ), ALL_CAT),
])

pipeline_wx = Pipeline([
    ("preprocessor", preprocessor_wx),
    ("regressor", HistGradientBoostingRegressor(
        max_iter=200, max_depth=5, learning_rate=0.05,
        categorical_features=cat_indices_wx,
        random_state=42,
    )),
])

model_wx = TransformedTargetRegressor(
    regressor=pipeline_wx,
    func=np.log1p,
    inverse_func=np.expm1,
)

X_train_wx = train_df[ALL_NUM + ALL_CAT]
X_test_wx  = test_df[ALL_NUM + ALL_CAT]

model_wx.fit(X_train_wx, y_train)

y_pred_wx = model_wx.predict(X_test_wx)
mae_wx    = mean_absolute_error(y_test, y_pred_wx)
rmse_wx   = np.sqrt(mean_squared_error(y_test, y_pred_wx))
r2_wx     = r2_score(y_test, y_pred_wx)

print(f"Weather   →  MAE: {mae_wx:.2f} h   RMSE: {rmse_wx:.2f} h   R²: {r2_wx:.3f}")

## 18. Baseline vs weather-enriched comparison

In [ ]:
results = pd.DataFrame({
    "Model":   ["Baseline (kinematic)", "Weather-enriched"],
    "MAE (h)": [mae_base, mae_wx],
    "RMSE (h)":[rmse_base, rmse_wx],
    "R²":      [r2_base, r2_wx],
})
results["ΔMAE"]  = results["MAE (h)"]  - results["MAE (h)"].iloc[0]
results["ΔR²"]   = results["R²"]       - results["R²"].iloc[0]
display(results.round(3))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle("Baseline vs Weather-Enriched Model", fontsize=12)

colours = ["steelblue", "darkorange"]
labels  = ["Baseline", "Weather"]

for ax, metric, ylabel in zip(axes,
                               ["MAE (h)", "RMSE (h)", "R²"],
                               ["MAE (hours)", "RMSE (hours)", "R²"]):
    ax.bar(labels, results[metric], color=colours, alpha=0.8, edgecolor="white")
    for i, v in enumerate(results[metric]):
        ax.text(i, v + results[metric].max() * 0.02, f"{v:.3f}",
                ha="center", fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 19. Feature importance — weather-enriched model

In [ ]:
rng        = np.random.default_rng(42)
sample_idx = rng.choice(len(X_test_wx), size=min(10_000, len(X_test_wx)), replace=False)

result_wx = permutation_importance(
    model_wx,
    X_test_wx.iloc[sample_idx],
    y_test.iloc[sample_idx],
    n_repeats=10,
    random_state=42,
    n_jobs=-1,
)

importances_wx = pd.Series(
    result_wx.importances_mean,
    index=ALL_NUM + ALL_CAT,
).sort_values(ascending=True)  # ascending for horizontal barh

# Mark which features are new (weather) vs baseline
weather_set  = set(WEATHER_NUM + WEATHER_BIN + WEATHER_ORD)
bar_colours  = ["darkorange" if f in weather_set else "steelblue"
                for f in importances_wx.index]

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(range(len(importances_wx)), importances_wx.values,
        color=bar_colours, alpha=0.8)
ax.set_yticks(range(len(importances_wx)))
ax.set_yticklabels(importances_wx.index, fontsize=9)
ax.set_xlabel("Mean permutation importance (decrease in R²)")
ax.set_title("Feature importances — weather-enriched model")
ax.grid(axis="x", alpha=0.3)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="steelblue",  alpha=0.8, label="Baseline features"),
    Patch(color="darkorange", alpha=0.8, label="Weather features"),
], fontsize=9)

plt.tight_layout()
plt.show()

print("\nAll feature importances:")
display(importances_wx.sort_values(ascending=False).round(4).to_frame("importance"))

## 20. Save models

In [ ]:
models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model_wx,   models_dir / "delay_predictor_weather.joblib")
joblib.dump(model_base, models_dir / "delay_predictor.joblib")

print("Saved:")
print(f"  {models_dir / 'delay_predictor_weather.joblib'}  ← weather-enriched model")
print(f"  {models_dir / 'delay_predictor.joblib'}          ← baseline (overwritten)")